In [113]:
from snowflake.snowpark import Session
import pandas as pd

In [114]:
# 1. Define your connection parameters
connection_parameters = {
    "account": "rt27428.eu-central-1",               # Wise Snowflake account
    "user": "priscaadenike.adeoti@transferwise.com",        # your Wise email
    "authenticator": "EXTERNALBROWSER",              # opens Okta in browser
    "warehouse": "ANALYSTS",
    "database": "ANALYTICS_DB",
}

# 2. Create a Snowflake session
session = Session.builder.configs(connection_parameters).create()


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://transferwise.okta-emea.com/app/snowflake/exk4istpb5gZUyV8u0i7/sso/saml?SAMLRequest=nZJPc9owEMW%2Fikc927IdpyYaTIaGppBAS8DkwE3YC2iwJUcrY%2Binr8yfTnpIDr155Lf7e7tvu%2FeHsnD2oFEomZDA84kDMlO5kJuELNJHt0McNFzmvFASEnIEJPe9LvKyqFi%2FNls5g7ca0Di2kUTW%2FkhIrSVTHAUyyUtAZjI270%2FGLPR8xhFBG4sjl5IchWVtjakYpU3TeM2Np%2FSGhr7vU%2F%2BOWlUr%2BULeIarPGZVWRmWquJYc7EwfIALqRy3CKixhein8JuR5BZ9RVmcRsmGaTt3pr3lKnP51ugclsS5Bz0HvRQaL2fhsAK2DWRrGUdjxoHYzkEbzwg08lKpZF3wHmSqr2tjGnv2ia8hpoTbCrms0SEi1EzkMhp26HPdf4h8yHX9fZsfh781%2Buxo8T56f5hBNpLp7e4kXT5nKiPN6DTdswx0h1jCSbaTGPvnhV9e%2FcYM4DQLmR%2Bw28qI4XhJnYCMVkptT5dW3dSpxDboRCJ7aGe5CCfxkk1cV%2FTsBhcMuEmiq1e1muTi%2BdmpfxBRR0TY7cj4fdrKie%2F%2BxlC593%2BByjD9tPqPBVBUiOzqPSpfcfBxf4AWnF5G765OUQclF0c9zDYg2xqJQzYMGbuzNG10Dob0z9d%2Br7%2F0B&RelayState=ver%3A3-hint%3A11514986436542-ETMsDgAAAZz7eFnNABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAECHuiqaGN%2BekD42Qmgo6Lg8AAACgCyTz2

In [ ]:
# Comprehensive analysis for all rules
all_rules_query = """
WITH rule_mapping AS (
    SELECT 
        ID as case_id,
        REFERENCE_ID as profile_id,
        TIME_CREATED as created_time,
        CASE
            WHEN UPPER(metadata) LIKE UPPER('%PaymentForRequestRecentlyPublished%') THEN 'PaymentForRequestRecentlyPublished'
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_UniqueCardsRefusedLast24HoursThresholdBreached_PauseMerchant%') THEN 'UniqueCardsRefusedLast24HoursThresholdBreached'
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_UniqueCardsRefusedLast24HoursThresholdBreached_Reusable_PauseMerchant%')THEN 'UniqueCardsRefusedLast24HoursThresholdBreached_Reusable'
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_UniqueCardsRefusedLast24HoursThresholdBreached_Wisetag_PauseMerchant%')THEN 'UniqueCardsRefusedLast24HoursThresholdBreached_Wisetag'
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_emailAgeScoreAboveThreshold_ReviewMerchant%')THEN 'emailAgeScoreAboveThreshold'
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_HighRiskIssuerRefusalReason_ReviewMerchant%')THEN 'HighRiskIssuerRefusalReason'
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_PAYFAC_POST_AUTH_PAYFAC_SUB_MERCHANTAbuserFlag_AlertSlack%')THEN 'AbuserFlag'
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_AverageDailyPayFacTransactionExceedsAverageDailyTransfer_PauseMerchant%')THEN 'AverageDailyPayFacTransactionExceedsAverageDailyTransfer'
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_AttemptedAmountExceedsTransferAmount_PauseMerchant%') THEN 'AttemptedAmountExceedsTransferAmount' 
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_Bin6AttemptedVolumeSpike_ReviewMerchant%') THEN 'Bin6AttemptedVolumeSpike'
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_ProtonmailEmailDomain_PauseMerchant%') THEN 'ProtonmailEmailDomain'
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_IssuerAndMerchantConcentration_ReviewMerchant%') THEN 'IssuerAndMerchantConcentration'
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_IssuerIcardADPattern_DeclineAndOffboard%') THEN 'IssuerIcardADPattern'
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_LTVHigherThresholdBreached_PauseMerchant%') THEN 'LTVHigherThresholdBreached'
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_LTVLowerThresholdBreached_ReviewMerchant%') THEN 'LTVLowerThresholdBreached'
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_PayerDisputeThreshold_ReviewMerchant') THEN 'PayerDisputeThreshold'
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_WithdrawalRule_PauseMerchant') THEN 'WithdrawalRule'
            ELSE 'Transaction_amount'
        END AS rule_group
    FROM DDCASE.DD_CASE
    WHERE TIME_CREATED >= DATEADD(month, -6, CURRENT_DATE())
        AND case_type = 'ACQUIRING_MERCHANT_REVIEW'
        AND state = 'CLOSED'
),

rule_cases_deduplicated AS (
    -- Get the FIRST case for each profile-rule combination
    SELECT 
        rule_group,
        profile_id,
        MIN(created_time) as created_time
    FROM rule_mapping
    GROUP BY rule_group, profile_id
),

merchant_payments AS (
    SELECT 
        rc.rule_group,
        rc.profile_id,
        p.CARD_ATTEMPT_ID,
        p.ATTEMPTED_GBP_AMOUNT_GROSS,
        p.FRAUD_FLAG,
        p.CHARGEBACK_AMOUNT_GBP,
        m.date_offboarded,
    FROM rule_cases_deduplicated rc
    LEFT JOIN RPT_PRODUCT.PAY_WITH_CARD_MERCHANTS m ON m.profile_id = rc.profile_id
    LEFT JOIN RPT_PRODUCT.PAY_WITH_CARD_PAYMENTS p ON rc.profile_id = p.PROFILE_ID
        AND p.ATTEMPT_TIME > rc.created_time
        AND p.ATTEMPT_TIME <= m.date_offboarded
        AND p.successful_transaction_flag = 'success'
    
)

SELECT 
    rule_group,
    COUNT(DISTINCT profile_id) as unique_merchants,
    COALESCE(COUNT(CARD_ATTEMPT_ID), 0)  as total_payments,
    COALESCE(SUM(CASE WHEN FRAUD_FLAG = TRUE THEN 1 ELSE 0 END), 0) as NOF_payments,
    COALESCE(SUM(CASE WHEN FRAUD_FLAG = TRUE THEN ATTEMPTED_GBP_AMOUNT_GROSS ELSE 0 END), 0) as NOF_amount_gbp,
    COALESCE(SUM(CASE WHEN CHARGEBACK_AMOUNT_GBP IS NOT NULL THEN 1 ELSE 0 END), 0) as chargeback_payments,
    COALESCE(SUM(COALESCE(CHARGEBACK_AMOUNT_GBP, 0)), 0) as chargeback_amount_gbp,
    COALESCE(
        COUNT(DISTINCT CASE WHEN CHARGEBACK_AMOUNT_GBP IS NOT NULL OR FRAUD_FLAG = TRUE THEN profile_id END),
        0
    ) as problematic_merchants,
    COALESCE(
        SUM(CASE WHEN FRAUD_FLAG = TRUE THEN ATTEMPTED_GBP_AMOUNT_GROSS ELSE 0 END), 0
    ) + COALESCE(
        SUM(COALESCE(CHARGEBACK_AMOUNT_GBP, 0)), 0
    ) as total_fraud_gbp,
    CASE 
        WHEN COUNT(DISTINCT CASE WHEN CHARGEBACK_AMOUNT_GBP IS NOT NULL OR FRAUD_FLAG = TRUE THEN profile_id END) > 0
        THEN
            (
                COALESCE(SUM(CASE WHEN FRAUD_FLAG = TRUE THEN ATTEMPTED_GBP_AMOUNT_GROSS ELSE 0 END), 0) +
                COALESCE(SUM(COALESCE(CHARGEBACK_AMOUNT_GBP, 0)), 0)
            )
            / NULLIF(
                COUNT(DISTINCT CASE WHEN CHARGEBACK_AMOUNT_GBP IS NOT NULL OR FRAUD_FLAG = TRUE THEN profile_id END),
                0
            )
        ELSE 0
END AS avg_fraud_per_problematic_merchant  -- This will give an idea o if we do not review a merchant by this rule and they turn out to be bad, how much we might lose on average
FROM merchant_payments
GROUP BY rule_group
ORDER BY total_fraud_gbp DESC
"""

print("=" * 100)
print("COMPREHENSIVE LOSS ANALYSIS FOR ALL RULES (Last 6 Months)")
print("=" * 100)
print("\n⏳ Executing query... This may take a moment...\n")

df_all_rules = session.sql(all_rules_query).to_pandas()

print("✓ Query complete!\n")
print("=" * 100)

COMPREHENSIVE LOSS ANALYSIS FOR ALL RULES (Last 6 Months)

⏳ Executing query... This may take a moment...

✓ Query complete!



In [117]:
# Display the results in a clear format
print("=" * 100)
print("RESULTS: LOSS ESTIMATION BY RULE")
print("=" * 100 + "\n")

if len(df_all_rules) > 0:
    # Normalise column names to lowercase (Snowflake -> pandas is usually lowercase)
    df_all_rules.columns = [c.lower() for c in df_all_rules.columns]

    # Format the dataframe for better display
    df_display = df_all_rules.copy()
    df_display['nof_amount_gbp'] = df_display['nof_amount_gbp'].apply(lambda x: f'£{x:,.2f}')
    df_display['chargeback_amount_gbp'] = df_display['chargeback_amount_gbp'].apply(lambda x: f'£{x:,.2f}')
    df_display['total_fraud_gbp'] = df_display['total_fraud_gbp'].apply(lambda x: f'£{x:,.2f}')
    df_display['avg_fraud_per_problematic_merchant'] = round(df_display['avg_fraud_per_problematic_merchant'], 2)
    
    print(df_display.to_string(index=False))
    
    print("\n" + "=" * 100)
    print("SUMMARY STATISTICS")
    print("=" * 100 + "\n")
    
    total_merchants = df_all_rules['unique_merchants'].sum()
    total_problematic_merchants = df_all_rules['problematic_merchants'].sum()
    total_payments = df_all_rules['total_payments'].sum()
    total_NOF_amt = df_all_rules['nof_amount_gbp'].sum()
    total_cb_amt = df_all_rules['chargeback_amount_gbp'].sum()
    total_fraud = df_all_rules['total_fraud_gbp'].sum()
    
    print(f"Total Unique Merchants: {total_merchants:,}")
    print(f"Total Payments Analyzed: {total_payments:,}")
    print(f"Total NOF Amount: £{total_NOF_amt:,.2f}")
    print(f"Total Chargeback Amount: £{total_cb_amt:,.2f}")
    print(f"TOTAL FRAUD (ALL RULES): £{total_fraud:,.2f}")
    print(f"Monthly Average Fraud: £{total_fraud/6:,.2f}")
    
    print("\n" + "=" * 100)
    
else:
    print("⚠️ No data returned for the specified rules")

print("\n✓ Results saved as 'df_all_rules'")

RESULTS: LOSS ESTIMATION BY RULE

                                              rule_group  unique_merchants  total_payments  nof_payments nof_amount_gbp  chargeback_payments chargeback_amount_gbp  problematic_merchants total_fraud_gbp  avg_fraud_per_problematic_merchant
                    AttemptedAmountExceedsTransferAmount               136              63             0          £0.00                    4            £45,978.38                      2      £45,978.38                            22989.19
                      PaymentForRequestRecentlyPublished                81              68             0          £0.00                    8            £44,307.08                      3      £44,307.08                            14769.03
AverageDailyPayFacTransactionExceedsAverageDailyTransfer               872            4522            34      £9,557.95                   63            £28,885.15                     34      £38,443.11                             1130.68
              

In [140]:
# Create the current state dataframe
import pandas as pd
import numpy as np


Avg_Cases_Per_Month = [23, 292, 32, 10, 27, 1, 9, 67, 7, 14, 14, 10, 20, 6, 77, 13, 67]
Avg_Handling_Time_Minutes = [18, 35, 23, 27, 19, 41, 132, 31, 12, 43, 17, 40, 35, 26, 28, 24, 13]
Current_Hours_Per_Month = [
    round((cases * aht) / 60,2)
    for cases, aht in zip(Avg_Cases_Per_Month, Avg_Handling_Time_Minutes)
]

# Input data
current_state = {
    'Rule_Name': [
        'AttemptedAmountExceedsTransferAmount',
        'AverageDailyPayFacTransactionExceedsAverageDailyTransfer',
        'Bin6AttemptedVolumeSpike',
        'HighRiskIssuerRefusalReason',
        'IssuerAndMerchantConcentration',
        'IssuerIcardADPattern',
        'LTVHigherThresholdBreached',
        'LTVLowerThresholdBreached',
        'PayerDisputeThreshold',
        'PaymentForRequestRecentlyPublished',
        'ProtonmailEmailDomain',
        'UniqueCardsRefusedLast24HoursThresholdBreached',
        'UniqueCardsRefusedLast24HoursThresholdBreached_Reusable',
        'UniqueCardsRefusedLast24HoursThresholdBreached_Wisetag',
        'WithdrawalRule',
        'emailAgeScoreAboveThreshold',
        'Transaction_amount',
        
    ],
    'per_case_Precision_%': [55.56, 37.91, 52.78, 37.5, 16.67, 100.0, 30.0, 26.98, 10.0, 50.0, 37.5, 44.44, 41.67, 71.43, 43.14, 33.33, 6.38],
    'Avg_Cases_Per_Month': Avg_Cases_Per_Month,
    'Avg_Handling_Time_Minutes': Avg_Handling_Time_Minutes,
    'Current_Review_Type': ['Standard', 'Standard', 'Standard', 'Standard', 'Standard', 'Standard', 'Deep', 'Deep', 'Standard', 'Standard', 'Standard', 'Standard', 'Standard', 'Standard', 'Standard', 'Standard', 'Standard'],
    'Current_Hours_Per_Month': Current_Hours_Per_Month
}

df_current = pd.DataFrame(current_state)



# Add loss data from our earlier analysis
 #first we define a helper function to get the average fraud per problematic merchant for each rule from our earlier analysis dataframe (df_display) as a value rather than series
def get_scalar(df, rule, col, default=0.0):
    s = df.loc[df["rule_group"] == rule, col]
    if s.empty:
        return default
    return float(s.iloc[0])  # numeric scalar


loss_mapping = {
    'AttemptedAmountExceedsTransferAmount': get_scalar(df_display, 'AttemptedAmountExceedsTransferAmount', 'avg_fraud_per_problematic_merchant'),
    'PaymentForRequestRecentlyPublished': get_scalar(df_display, 'PaymentForRequestRecentlyPublished', 'avg_fraud_per_problematic_merchant'),
    'AverageDailyPayFacTransactionExceedsAverageDailyTransfer': get_scalar(df_display, 'AverageDailyPayFacTransactionExceedsAverageDailyTransfer', 'avg_fraud_per_problematic_merchant'),
    'UniqueCardsRefusedLast24HoursThresholdBreached_Wisetag': get_scalar(df_display, 'UniqueCardsRefusedLast24HoursThresholdBreached_Wisetag', 'avg_fraud_per_problematic_merchant'),
    'Bin6AttemptedVolumeSpike': get_scalar(df_display, 'Bin6AttemptedVolumeSpike', 'avg_fraud_per_problematic_merchant'),
    'WithdrawalRule': get_scalar(df_display, 'WithdrawalRule', 'avg_fraud_per_problematic_merchant'),
    'UniqueCardsRefusedLast24HoursThresholdBreached': get_scalar(df_display, 'UniqueCardsRefusedLast24HoursThresholdBreached', 'avg_fraud_per_problematic_merchant'),
    'UniqueCardsRefusedLast24HoursThresholdBreached_Reusable': get_scalar(df_display, 'UniqueCardsRefusedLast24HoursThresholdBreached_Reusable', 'avg_fraud_per_problematic_merchant'),
    'emailAgeScoreAboveThreshold': get_scalar(df_display, 'emailAgeScoreAboveThreshold', 'avg_fraud_per_problematic_merchant'),
    'HighRiskIssuerRefusalReason': get_scalar(df_display, 'HighRiskIssuerRefusalReason', 'avg_fraud_per_problematic_merchant'),
    'ProtonmailEmailDomain': get_scalar(df_display, 'ProtonmailEmailDomain', 'avg_fraud_per_problematic_merchant'),
    'LTVLowerThresholdBreached': get_scalar(df_display, 'LTVLowerThresholdBreached', 'avg_fraud_per_problematic_merchant'),
    'IssuerAndMerchantConcentration': get_scalar(df_display, 'IssuerAndMerchantConcentration', 'avg_fraud_per_problematic_merchant'),
    'LTVHigherThresholdBreached': get_scalar(df_display, 'LTVHigherThresholdBreached', 'avg_fraud_per_problematic_merchant'),
    'PayerDisputeThreshold': get_scalar(df_display, 'PayerDisputeThreshold', 'avg_fraud_per_problematic_merchant'),
    'IssuerIcardADPattern': get_scalar(df_display, 'IssuerIcardADPattern', 'avg_fraud_per_problematic_merchant'),
    'Transaction_amount': get_scalar(df_display, 'Transaction_amount', 'avg_fraud_per_problematic_merchant'),
}


df_current['avg_fraud_volume_per_problematic_merchant'] = (df_current['Rule_Name'].map(loss_mapping))   #for each rule, map the average fraud volume (CB + NOF) per problematic merchant from our earlier analysis
df_current['monthly_avg_fraud_volume_per_problematic_merchant'] = df_current['avg_fraud_volume_per_problematic_merchant'] / 6  #on average what is the loss per merchant per month

# Review time constants (in minutes)
LIGHT_REVIEW_MIN = 12.4
STANDARD_REVIEW_MIN = 35.15
DEEP_REVIEW_MIN = 47.97

print("=" * 120)
print("CURRENT STATE ANALYSIS")
print("=" * 120 + "\n")

# Calculate current weekly hours
current_monthly_hours = df_current['Current_Hours_Per_Month'].sum()  
current_weekly_hours = current_monthly_hours * 12 / 52 + 20 # Adding 20 hours for NOF, CB
Target_Weekly_Hours = 100 #Enter the Capacity Hours / Week from the sheet (currently H34)
print(f"Current Total Monthly Hours: {current_monthly_hours:.2f}")
print(f"Current Weekly Hours: {current_weekly_hours:.2f}")
print(f"Target Weekly Hours: {Target_Weekly_Hours:.2f}")
print(f"Reduction Needed: {current_weekly_hours - Target_Weekly_Hours} hours/week ({(current_weekly_hours - Target_Weekly_Hours)/current_weekly_hours * 100:.1f}%)")

print("\n" + "=" * 120)

CURRENT STATE ANALYSIS

Current Total Monthly Hours: 349.64
Current Weekly Hours: 100.69
Target Weekly Hours: 100.00
Reduction Needed: 0.6861538461538572 hours/week (0.7%)



In [180]:
# Create scoring system to determine optimal review type
# Factors: Loss risk, Precision, Volume

def calculate_priority_score(row):
    """
    Calculate priority score based on:
    - Monthly loss (higher = higher priority)
    - per case precision (higher = higher confidence, but also means we're catching them)
    - Volume (higher volume = more impact)
    """
    
    # Normalize loss (0-100 scale) so there is not too much skew from the highest loss rule and we can still differentiate between the lower loss rules
    max_loss = df_current['monthly_avg_fraud_volume_per_problematic_merchant'].max()  # Max average monthlyloss per problematic merchant
    loss_score = min((row['monthly_avg_fraud_volume_per_problematic_merchant'] / max_loss) * 100, 100) if max_loss > 0 else 0
    
    # Per case precision score 
    precision_score = row['per_case_Precision_%']
    
    # Volume score (normalized as well)
    max_volume = max(Avg_Cases_Per_Month)  # Max cases per month across rules
    volume_score = (row['Avg_Cases_Per_Month'] / max_volume) * 100
    
    # Weighted score: Loss (50%), Precision (30%), Volume (20%)
    priority_score = (loss_score * 0.5) + (precision_score * 0.3) + (volume_score * 0.2)
    
    return priority_score

df_current['Priority_Score'] = df_current.apply(calculate_priority_score, axis=1)

# Sort by priority
df_current_sorted = df_current.sort_values(by = 'Priority_Score', ascending=False)

print("\nRULE PRIORITIZATION")
print("=" * 120)
print(f"\n{'Rule':<60} {'Loss/mo':<10} {'Precision':<10} {'Num_Cases':<10} {'Score':<7} {'Current_Review_Type':<10}")
print(f"{'─'*60} {'─'*10} {'─'*10} {'─'*10} {'─'*7} {'─'*10}")

for _, row in df_current_sorted.iterrows():
    print(f"{row['Rule_Name']:<60} £{row['monthly_avg_fraud_volume_per_problematic_merchant']:>8,.0f}  {row['per_case_Precision_%']:>9.1f}%   {int(row['Avg_Cases_Per_Month']):>5}   {row['Priority_Score']:>5.1f}   {row['Current_Review_Type']:>10}")

print("\n" + "=" * 120)


RULE PRIORITIZATION

Rule                                                         Loss/mo    Precision  Num_Cases  Score   Current_Review_Type
──────────────────────────────────────────────────────────── ────────── ────────── ────────── ─────── ──────────
AttemptedAmountExceedsTransferAmount                         £   3,832       55.6%      23    68.2     Standard
PaymentForRequestRecentlyPublished                           £   2,462       50.0%      14    48.1     Standard
AverageDailyPayFacTransactionExceedsAverageDailyTransfer     £     188       37.9%     292    33.8     Standard
IssuerIcardADPattern                                         £       0      100.0%       1    30.1     Standard
Transaction_amount                                           £   1,446        6.4%      67    25.4     Standard
UniqueCardsRefusedLast24HoursThresholdBreached_Wisetag       £      16       71.4%       6    22.1     Standard
Bin6AttemptedVolumeSpike                                     £     294 

### New review type assignment
Using this score together with the current review type, we can assign a new review type.  
Rules with higer score get a better review.
Review priority flows as:
> **Deep → Standard → Light**

Ideally, I like to keep **LTVHigherThresholdBreached** as **Deep** review due to the original reason this rule was created.

In [16]:
# Check current review type distribution with correct column name
print("Current Review Type Distribution:")
print("="*80)
print(df_current_sorted['Current_Review_Type'].value_counts())
print(f"\nTotal rules: {len(df_current_sorted)}")

Current Review Type Distribution:
Current_Review_Type
Standard    13
Deep         2
Name: count, dtype: int64

Total rules: 15
